# Full-map MPC collection and fitting rounds

Use a Colab GPU. This run uses the six goals and eight square bands from the earlier full-map experiment, with three goals in each room. The square half-widths are 6, 12, 18, 24, 30, 36, 42, and 48. The last square covers the reset-safe region for every goal. Round 1 reuses the earlier start positions and planner seeds, and later rounds sample fresh starts in the same band.

Every rollout starts with one-macro MPC. Each band gets three rounds, with up to six starts per goal in each round. A nonempty round fits for 500 updates using its new pairs and the earlier replay. The next round collects with that cost. One eligible pair is enough to fit. A zero-pair round carries forward the current cost, including raw MSE if fitting has yet to start.

Only successful episodes supply temporal pairs, with no shortest-path progress filter. Shortest legal path distance is used for held-out evaluation. The loss scale is derived from the first collected batch that supplies training pairs.

The default schedule contains 810 rollout attempts. Set `LAST_BAND` lower to run a first segment. To continue, set `RESUME_RUN_ID` to the printed ID and raise `LAST_BAND` while keeping the round count, start count, and runtime fixed. The results are stored in a separate Drive folder.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
ROUNDS_PER_BAND=3
LAST_BAND=8
PLOT_GOALS=[1,3]         # Collection goals, one per room
PLOT_BANDS=[4,6,8]       # Applies to trajectory plots and heatmaps
HEATMAP_GOALS=[0,4]      # Held-out evaluation goals, one per room
DRAW_COST_MAPS=True
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_mpc_repeated_full_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-full-mpc-repeated')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='dd406f228e0357fce569e5791536f0173966963a'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the frozen encoder, world model and renderer

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_square',operation,
          '--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL,'--rounds-per-band',ROUNDS_PER_BAND]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Collect and fit

Each invocation runs one band and saves each completed round separately. Use the same run ID after an interruption.

In [ ]:
assert ROUNDS_PER_BAND>=2 and 1<=LAST_BAND<=8
for band in range(1,LAST_BAND+1):
    run_band(band)

## Check collection by round and route

A crossing counts when the ball passes through the doorway and reaches the other room. A rollout can cross and still fail to reach the goal. Square entry measures whether the ball entered the previous square by action 5 or at any later point. Crossing rates apply to opposite-room starts. Doorway pairs are retained simulator transitions along the opening.

In [ ]:
from IPython.display import Image, display
import numpy as np
import pandas as pd

def read_json(path):
    return json.loads(path.read_text())

manifest=read_json(OUTPUT/'experiment.json')['manifest']
stages={}
for band in range(1,LAST_BAND+1):
    for number in range(1,ROUNDS_PER_BAND+1):
        path=OUTPUT/f'band_{band}_round_{number}'/'complete.json'
        if path.is_file():
            stages[band,number]=read_json(path)
if not stages:
    raise RuntimeError('Finish at least one collection round first.')
print('Collection goals:',[tuple(round(v,2) for v in g) for g in manifest['goals']])

rows=[]
for (band,number),stage in stages.items():
    for row in stage['collection']:
        route=row['case']['route']
        cross_room=route in ('left_to_right','right_to_left')
        diag=row['boundary']
        rows.append(dict(band=band,round=number,goal=row['case']['goal_index'],route=route,
            success=row['success'],crossed=row['reached_other_room'] if cross_room else np.nan,
            entered_by_5=diag['entered_by_five_actions'] if band>1 else np.nan,
            entered=diag['first_entry_step'] is not None if band>1 else np.nan,
            pairs=row['retained_pairs'],doorway_pairs=row['retained_doorway_pairs']))
data=pd.DataFrame(rows)

def summarize(grouped):
    return grouped.agg(attempts=('success','size'),successes=('success','sum'),
        success_rate=('success','mean'),crossed=('crossed',lambda x:x.sum(min_count=1)),
        crossing_rate=('crossed','mean'),entered_by_5=('entered_by_5',lambda x:x.sum(min_count=1)),
        entered_ever=('entered',lambda x:x.sum(min_count=1)),
        new_pairs=('pairs','sum'),doorway_pairs=('doorway_pairs','sum')).round(3)

rounds=summarize(data.groupby(['band','round']))
rounds['cumulative_pairs']=rounds.new_pairs.cumsum()
rounds['updates']=[stages[key]['updates'] for key in rounds.index]
display(rounds)
by_route=summarize(data.groupby(['band','round','route']))
print('Opposite-room starts by direction')
display(by_route[by_route.index.get_level_values('route').isin(['left_to_right','right_to_left'])])
rounds.to_csv(OUTPUT/'round_summary.csv')
by_route.to_csv(OUTPUT/'round_summary_by_route.csv')

## See the collection paths

The wall is gray and the goal is a star. The solid and dashed outlines mark the inner and outer square edges. Green paths show successes and small white dots mark retained training observations. Failed paths show their first five actions in red plus a dashed start-to-end segment. The failed segment summarizes displacement and may visually cross a wall.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

def wall_overlay(ax):
    for y,height in ((4,22),(34,26)):
        ax.add_patch(Rectangle((29,y),6,height,facecolor='#555',alpha=.75,zorder=2))

def square_edges(ax, goal, band):
    gx,gy=goal
    for radius,style in zip(band,('-','--')):
        if radius:
            ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,fill=False,
                                   edgecolor='#db9123',linestyle=style,linewidth=1.2,zorder=3))

plot_dir=OUTPUT/'round_trajectories'
plot_dir.mkdir(exist_ok=True)
for band in PLOT_BANDS:
    for goal_index in PLOT_GOALS:
        groups=[]
        for (b,number),stage in stages.items():
            if b==band:
                subset=[r for r in stage['collection'] if r['case']['goal_index']==goal_index]
                if subset:
                    groups.append((number,subset))
        if not groups:
            continue
        goal=manifest['goals'][goal_index]
        fig,axes=plt.subplots(1,len(groups),figsize=(4.65*len(groups),5),squeeze=False,layout='constrained')
        for ax,(number,episodes) in zip(axes[0],groups):
            wall_overlay(ax)
            square_edges(ax,goal,manifest['bands'][band-1])
            for row in sorted(episodes,key=lambda r:r['success']):
                path=np.asarray(row['boundary']['positions'])
                color='#218c4a' if row['success'] else '#c34049'
                if row['success']:
                    ax.plot(*path.T,color=color,alpha=.72,zorder=4)
                    retained=np.asarray(row['retained_positions']).reshape(-1,2)
                    if len(retained):
                        ax.scatter(*retained.T,s=14,facecolors='white',edgecolors=color,linewidths=.65,zorder=5)
                else:
                    ax.plot(*path[:6].T,color=color,alpha=.6,zorder=4)
                    ax.plot(*path[[0,-1]].T,color=color,linestyle='--',alpha=.4,zorder=4)
                ax.scatter(*path[0],s=30,facecolors='none',edgecolors=color,zorder=6)
                ax.scatter(*path[-1],s=30,color=color,marker='x',zorder=6)
            ax.scatter(*goal,marker='*',s=125,c='black',edgecolors='white',zorder=7)
            successes=sum(row['success'] for row in episodes)
            cross=sum(row['reached_other_room'] for row in episodes)
            ax.set(title=f'Round {number} · {successes}/{len(episodes)} success · {cross} crossed',
                   xlim=(4,60),ylim=(4,60),aspect='equal',xlabel='Ball x',ylabel='Ball y')
        fig.suptitle(f'Collection goal {goal_index} · Band {band}')
        fig.legend(handles=[
            Line2D([],[],color='#218c4a',label='Success'),
            Line2D([],[],color='#c34049',label='Failed first five'),
            Line2D([],[],color='#c34049',ls='--',label='Failed start to end'),
            Line2D([],[],marker='o',mfc='white',mec='#218c4a',ls='none',label='Training observation'),
        ],loc='upper center',bbox_to_anchor=(.5,-.005),ncol=4,frameon=False)
        path=plot_dir/f'goal_{goal_index}_band_{band}.png'
        fig.savefig(path,dpi=135,bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))

## Inspect the cost geometry on held-out goals

The table compares ordering against shortest legal path distance. It separates the whole map from positions in the room opposite the goal. The maps use median-normalized latent MSE on a common scale for each goal. Each row starts with the incoming cost and continues through that band's rounds. The panels mark zero-update rounds as carried forward.

In [ ]:
if DRAW_COST_MAPS:
    finished=[b for b in range(1,LAST_BAND+1) if (b,ROUNDS_PER_BAND) in stages]
    if not finished:
        raise RuntimeError('Finish all rounds of at least one band to compare costs.')
    last=max(finished)
    shell(command('compare',last),cwd=REPO)
    report=read_json(OUTPUT/'comparison.json')
    rows=[]
    for stage in report['stages']:
        for i,goal in enumerate(stage['by_goal']):
            for cohort in ('full_map','opposite_room','local','local_opposite_room'):
                for method,result in goal['metrics'][cohort].items():
                    rows.append(dict(band=stage['band'],round=stage['round'],goal=i,
                        cohort=cohort,method=method,**result))
    metrics=pd.DataFrame(rows)
    metrics[['correct','uphill']]*=100
    display(metrics.pivot_table(index=['band','round'],columns=['cohort','method'],values='correct').round(2))
    metrics.to_csv(OUTPUT/'heldout_ranking_by_round.csv',index=False)

    axis=report['axis']
    shape=(len(axis),len(axis))
    plotted=[b for b in PLOT_BANDS if b in finished]
    plots=OUTPUT/'heatmaps'
    plots.mkdir(exist_ok=True)
    checkpoints={(s['band'],s['round']):s for s in report['stages']}
    for index in HEATMAP_GOALS if plotted else []:
        goal=report['goals'][index]
        maps={(0,0):np.asarray(report['raw_grid'][index],dtype=float).reshape(shape)}
        maps.update({key:np.asarray(s['by_goal'][index]['grid_costs'],dtype=float).reshape(shape)
                     for key,s in checkpoints.items()})
        medians={key:float(np.nanmedian(cost)) for key,cost in maps.items()}
        scaled={key:cost/max(medians[key],1e-9) for key,cost in maps.items()}
        vmax=max(float(np.nanmax(cost)) for cost in scaled.values())
        cols=ROUNDS_PER_BAND+1
        fig,axes=plt.subplots(len(plotted),cols,figsize=(3.4*cols,3.8*len(plotted)),
                              squeeze=False,layout='constrained')
        for row,band in enumerate(plotted):
            incoming=(0,0) if band==1 else (band-1,ROUNDS_PER_BAND)
            keys=[incoming,*[(band,n) for n in range(1,cols)]]
            lo,hi=manifest['bands'][band-1]
            for number,(ax,key) in enumerate(zip(axes[row],keys)):
                mesh=ax.pcolormesh(axis,axis,scaled[key],shading='nearest',vmin=0,vmax=vmax,cmap='viridis')
                wall_overlay(ax)
                square_edges(ax,goal,(lo,hi))
                ax.scatter(*goal,marker='*',s=82,c='black',edgecolor='white',zorder=5)
                title='Incoming raw MSE' if key==(0,0) else 'Incoming' if not number else f'After round {number}'
                if number and not checkpoints[key]['updates']:
                    title+=' (carried forward)'
                ax.set(title=f'{title}\nMedian MSE {medians[key]:.3g}',xlim=(4,60),ylim=(4,60),
                       aspect='equal',xlabel='Ball x')
                if not number:
                    ax.set_ylabel(f'Band {band} ({lo}–{hi})\nBall y')
        fig.colorbar(mesh,ax=axes.ravel().tolist(),shrink=.6,label='MSE divided by map median')
        fig.suptitle(f'Held-out goal {index} {tuple(goal)}')
        path=plots/f'goal_{index}_rounds.png'
        fig.savefig(path,dpi=120,bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))